In [49]:
## Import modules

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from sqlalchemy import create_engine, URL, text
from getpass import getpass
from sqlalchemy import Numeric

In [50]:
base_url = "https://www.framesdirect.com/eyeglasses/?p={current_page}&rows=24&sortby=7&noreset=true"

pages = [current_page for current_page in range(1, 21)]


brand_name = []
product_name = []
product_price = []
color = []
rating = []

for current_page in pages:
    print(f"extracting page {current_page}....")
    


    url = base_url + str(current_page)
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    response = requests.get(url, headers=headers)

        
    if response.status_code != 200:
        print(f"Failed to fetch page {current_page}. Status code: {response.status_code}")
        continue

    soup = BeautifulSoup(response.content, 'html.parser')
    
    glass_listings = soup.find_all('div', class_="prod-holder")
    
    
    for glass in glass_listings:
        brand_el = glass.find(class_='catalog-name')
        brand = brand_el.text.strip() if brand_el else "Unknown Brand"
        
        # 2. Extract Product Name safely (Accounts for product-name or product_name)
        product_el = glass.find(class_='product-name') or glass.find(class_='product_name')
        product = product_el.text.strip() if product_el else "Unknown Product"
        
        # 3. Extract Product Price safely 
        price_el = glass.find(class_='prod-aslowas')
        price = price_el.text.strip() if price_el else "Unknown Price"
        
        # 4. Extract Color Name safely
        color_span = glass.find('span', id=re.compile(r'^dvSelectedColorName'))
        glass_color = color_span.text.strip() if color_span else "Unknown Color"
        
        # 5. Extract Star Rating safely
        popular_block = glass.find(class_='prod-popular')
        if popular_block:
            star_count = len(popular_block.find_all('i', class_='icon-star-solid'))
            rating_value = f"{star_count} Stars" if star_count > 0 else "No Reviews"
        else:
            rating_value = "No Reviews"
            
            
            # Append each glass
        brand_name.append(brand)
        product_name.append(product)
        product_price.append(price)
        color.append(glass_color)
        rating.append(rating_value)
            
    
glass_df = pd.DataFrame({
'brand_name': brand_name,
'product_name': product_name,
'product_price': product_price,
'color': color,
'rating': rating
})    

extracting page 1....
extracting page 2....
extracting page 3....
extracting page 4....
extracting page 5....
extracting page 6....
extracting page 7....
extracting page 8....
extracting page 9....
extracting page 10....
extracting page 11....
extracting page 12....
extracting page 13....
extracting page 14....
extracting page 15....
extracting page 16....
extracting page 17....
extracting page 18....
extracting page 19....
extracting page 20....


In [51]:
glass_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   brand_name     500 non-null    str  
 1   product_name   500 non-null    str  
 2   product_price  500 non-null    str  
 3   color          500 non-null    str  
 4   rating         500 non-null    str  
dtypes: str(5)
memory usage: 19.7 KB


In [35]:
glass_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   brand_name     500 non-null    str  
 1   product_name   500 non-null    str  
 2   product_price  500 non-null    str  
 3   color          500 non-null    str  
 4   rating         500 non-null    str  
dtypes: str(5)
memory usage: 19.7 KB


## Transformation

In [52]:
# Make a copy of original glass dataframe

clean_glass_df = glass_df.copy()

In [53]:
## Convert product price from string to float

# Convert product price from string to float
clean_glass_df["product_price"] = pd.to_numeric(
    clean_glass_df["product_price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
).astype("float64")


# Separate color into color and color_code
color_parts = clean_glass_df["color"].str.rsplit(
    " (", n=1, expand=True
)

clean_glass_df["color"] = color_parts[0].str.strip()
clean_glass_df["color_code"] = color_parts[1].str.removesuffix(")")

In [54]:
clean_glass_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   brand_name     500 non-null    str    
 1   product_name   500 non-null    str    
 2   product_price  480 non-null    float64
 3   color          500 non-null    str    
 4   rating         500 non-null    str    
 5   color_code     480 non-null    str    
dtypes: float64(1), str(5)
memory usage: 23.6 KB


In [55]:
clean_glass_df = clean_glass_df.dropna(
    subset=["brand_name", "product_name", "product_price", "color"],
    how="all"
).reset_index(drop=True)

In [56]:
clean_glass_df.head()

,brand_name,product_name,product_price,color,rating,color_code
0,Oakley,Airdrop,227.00,Satin Black,5 Stars,OX8046-01
1,Ray-Ban,RB7047,176.00,Matte Black,5 Stars,5196
2,Unknown Brand,Unknown Product,NaN,Unknown Color,No Reviews,NaN
3,RALPH by Ralph Lauren,RA7158U,105.75,Shiny Transparent Beige,No Reviews,6117
4,Oakley,Plank 2.0,93.50,Matte Translucent Blue,5 Stars,OX8046-


In [57]:
connection_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=getpass("Enter your PostgreSQL password: "),
    host="localhost",
    port=5432,
    database="glass_framesDB"
)

engine = create_engine(connection_url)

In [60]:
with engine.begin() as connection:
    clean_glass_df.to_sql(
        name = "glass_frames",
        con=connection,
        schema="public",
        if_exists="replace",
        index=False,
        dtype={
            "product_price": Numeric(10, 2)
        }
    )
    

print(f"Loaded {len(clean_glass_df)} rows")

Loaded 500 rows
